In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


# Set up PySpark

In [2]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("AI_Reasoning_Layer").getOrCreate()

# PATHS

In [8]:
# Career fit output
CAREER_FIT_PATH = "/content/drive/MyDrive/FYP1/Dataset/career_fit_report.csv"

# Writing analysis output
WRITING_SCORES_PATH = "/content/drive/MyDrive/FYP1/Dataset/writing_analysis_scores.csv"


DEMAND_BY_ROLE_PATH = "/content/drive/MyDrive/FYP1/Dataset/job_demand_forecast.csv"
SALARY_BY_ROLE_PATH = "/content/drive/MyDrive/FYP1/Dataset/salary_projection_report.csv"


EDU_PATH = "/content/drive/MyDrive/FYP1/Dataset/education_report.csv"

OUT_FOLDER = "/content/drive/MyDrive/FYP1/Dataset/ai_reasoning_output"


# LOAD REQUIRED INPUTS

In [9]:
career = (spark.read.option("header", True).option("inferSchema", True).csv(CAREER_FIT_PATH))

#  career output uses: student_id, Top1, Top2, Top3 (based on your notebook)

career = (career
          .withColumnRenamed("student_id", "Student_ID")
          .select("Student_ID", "Top1", "Top2", "Top3"))

writing = (spark.read.option("header", True).option("inferSchema", True).csv(WRITING_SCORES_PATH))

# writing output uses: student_id, clarity, structure, confidence, analytical, creativity
writing = (writing
           .withColumnRenamed("student_id", "Student_ID")
           .select("Student_ID", "clarity", "structure", "confidence", "analytical", "creativity"))

# OPTIONAL tables (role-level)
demand = None
salary = None

if DEMAND_BY_ROLE_PATH:

    demand = spark.read.option("header", True).option("inferSchema", True).csv(DEMAND_BY_ROLE_PATH)

if SALARY_BY_ROLE_PATH:

    salary = spark.read.option("header", True).option("inferSchema", True).csv(SALARY_BY_ROLE_PATH)

# BUILD CANDIDATE ROLE ROWS

In [10]:
# Create 3 rows per student: (Student_ID, role, rank)
candidates = (career
              .select("Student_ID",
                      F.array(
                          F.struct(F.lit(1).alias("rank"), F.col("Top1").alias("role")),
                          F.struct(F.lit(2).alias("rank"), F.col("Top2").alias("role")),
                          F.struct(F.lit(3).alias("rank"), F.col("Top3").alias("role"))
                      ).alias("role_list"))
              .withColumn("r", F.explode("role_list"))
              .select("Student_ID", F.col("r.rank").alias("career_rank"), F.col("r.role").alias("role"))
              .where(F.col("role").isNotNull() & (F.trim(F.col("role")) != ""))
             )

# Career-fit rank to score
# Rank1 -> 100, Rank2 -> 70, Rank3 -> 40
candidates = candidates.withColumn(
    "career_fit_score",
    F.when(F.col("career_rank") == 1, F.lit(100.0))
     .when(F.col("career_rank") == 2, F.lit(70.0))
     .otherwise(F.lit(40.0))
)


#  WRITING FIT SCORE (Explainable mapping by role)

In [11]:
# Join writing scores per student
cand = candidates.join(writing, on="Student_ID", how="left")

for c in ["clarity", "structure", "confidence", "analytical", "creativity"]:
    cand = cand.withColumn(c, F.coalesce(F.col(c).cast("double"), F.lit(50.0)))

# Create role groups (simple, explainable logic)

role = F.lower(F.col("role"))

cand = cand.withColumn(
    "role_group",
    F.when(role.rlike("business analyst|product|project"), F.lit("analysis_communication"))
     .when(role.rlike("data analyst|bi|marketing"), F.lit("analysis_reporting"))
     .when(role.rlike("software|developer|frontend|backend|full stack|mobile"), F.lit("engineering"))
     .when(role.rlike("ml|ai|data scientist"), F.lit("research_analytical"))
     .when(role.rlike("ui|ux|designer"), F.lit("creative"))
     .otherwise(F.lit("general"))
)

# Compute writing_fit_score depending on group
cand = cand.withColumn(
    "writing_fit_score",
    F.when(F.col("role_group") == "analysis_communication",
           (F.col("clarity")*0.35 + F.col("structure")*0.25 + F.col("confidence")*0.40))
     .when(F.col("role_group") == "analysis_reporting",
           (F.col("clarity")*0.40 + F.col("structure")*0.30 + F.col("analytical")*0.30))
     .when(F.col("role_group") == "engineering",
           (F.col("structure")*0.40 + F.col("clarity")*0.30 + F.col("analytical")*0.30))
     .when(F.col("role_group") == "research_analytical",
           (F.col("analytical")*0.50 + F.col("structure")*0.25 + F.col("clarity")*0.25))
     .when(F.col("role_group") == "creative",
           (F.col("creativity")*0.60 + F.col("clarity")*0.20 + F.col("confidence")*0.20))
     .otherwise((F.col("clarity") + F.col("structure") + F.col("confidence"))/3.0)
)


# OPTIONAL: DEMAND & SALARY SCORES

In [12]:
# If you don't have these tables yet, we use safe defaults (50).
cand = cand.withColumn("demand_score", F.lit(50.0))
cand = cand.withColumn("salary_score", F.lit(50.0))
cand = cand.withColumn("demand_trend", F.lit("Unknown"))
cand = cand.withColumn("salary_band", F.lit("Unknown"))

if demand is not None:
    # Support either demand_score OR demand_trend
    dcols = [c.lower() for c in demand.columns]
    d = demand
    # standardize
    if "role" not in dcols:
        # if your column is named differently, rename it here
        pass

    if "demand_score" in dcols:
        d = d.select(F.col("role").alias("role_d"), F.col("demand_score").cast("double").alias("demand_score_in"))
        cand = cand.join(d, cand["role"] == d["role_d"], "left").drop("role_d")
        cand = cand.withColumn("demand_score", F.coalesce(F.col("demand_score_in"), F.col("demand_score"))).drop("demand_score_in")

    if "demand_trend" in dcols:
        d = d.select(F.col("role").alias("role_d2"), F.col("demand_trend").alias("demand_trend_in"))
        cand = cand.join(d, cand["role"] == d["role_d2"], "left").drop("role_d2")
        cand = cand.withColumn("demand_trend", F.coalesce(F.col("demand_trend_in"), F.col("demand_trend"))).drop("demand_trend_in")

if salary is not None:
    scols = [c.lower() for c in salary.columns]
    s = salary

    # Option A: salary_score exists
    if "salary_score" in scols:
        s = s.select(F.col("role").alias("role_s"), F.col("salary_score").cast("double").alias("salary_score_in"))
        cand = cand.join(s, cand["role"] == s["role_s"], "left").drop("role_s")
        cand = cand.withColumn("salary_score", F.coalesce(F.col("salary_score_in"), F.col("salary_score"))).drop("salary_score_in")

    # Option B: salary_min/mid/max exists -> convert to a score (simple normalization idea)
    elif ("salary_mid" in scols) or ("salary_min" in scols and "salary_max" in scols):
        # Use salary_mid if available; else average(min,max)
        if "salary_mid" in scols:
            s2 = s.select(F.col("role").alias("role_s2"), F.col("salary_mid").cast("double").alias("salary_mid"))
        else:
            s2 = s.select(F.col("role").alias("role_s2"),
                          ((F.col("salary_min").cast("double") + F.col("salary_max").cast("double"))/2.0).alias("salary_mid"))

        # Normalize salary_mid into 0-100 using min/max across roles (Spark only)
        stats = s2.agg(F.min("salary_mid").alias("mn"), F.max("salary_mid").alias("mx")).collect()[0]
        mn = float(stats["mn"]) if stats["mn"] is not None else 0.0
        mx = float(stats["mx"]) if stats["mx"] is not None else 1.0
        denom = (mx - mn) if (mx - mn) != 0 else 1.0

        s2 = s2.withColumn(
            "salary_score_in",
            F.when(F.col("salary_mid").isNull(), F.lit(50.0))
             .otherwise(((F.col("salary_mid") - F.lit(mn)) / F.lit(denom)) * 100.0)
        ).withColumn(
            "salary_band",
            F.concat(F.lit("mid="), F.round(F.col("salary_mid"), 0).cast("string"))
        )

        cand = cand.join(s2.select("role_s2", "salary_score_in", "salary_band"),
                         cand["role"] == s2["role_s2"], "left").drop("role_s2")
        cand = cand.withColumn("salary_score", F.coalesce(F.col("salary_score_in"), F.col("salary_score"))).drop("salary_score_in")


# FINAL WEIGHTED SCORE

In [13]:
# used a weighted evidence integration layer
W_CAREER = 0.40
W_DEMAND = 0.20
W_SALARY = 0.15
W_WRITING = 0.25

cand = cand.withColumn(
    "final_score",
    F.col("career_fit_score")*F.lit(W_CAREER) +
    F.col("demand_score")*F.lit(W_DEMAND) +
    F.col("salary_score")*F.lit(W_SALARY) +
    F.col("writing_fit_score")*F.lit(W_WRITING)
)


#  PICK BEST ROLE PER STUDENT

In [14]:
w = Window.partitionBy("Student_ID").orderBy(F.desc("final_score"))

ranked = cand.withColumn("rn", F.row_number().over(w))

best = ranked.filter(F.col("rn") == 1)

# Confidence label (simple XAI rule):
# - High: score >= 75
# - Medium: 55-74
# - Low: < 55
best = best.withColumn(
    "confidence_label",
    F.when(F.col("final_score") >= 75, F.lit("High"))
     .when(F.col("final_score") >= 55, F.lit("Medium"))
     .otherwise(F.lit("Low"))
)


In [15]:
from pyspark.sql import functions as F

# ---------- Make all column names unique ----------
def make_unique_columns(df):
    counts = {}
    new_names = []
    for c in df.columns:
        if c not in counts:
            counts[c] = 0
            new_names.append(c)
        else:
            counts[c] += 1
            new_names.append(f"{c}__dup{counts[c]}")
    return df.toDF(*new_names)

best = make_unique_columns(best)

# ---------- Fix salary_band ambiguity ----------
# If there was a duplicate, it will now be named salary_band__dup1
if "salary_band__dup1" in best.columns:
    best = best.withColumn(
        "salary_band_final",
        F.coalesce(F.col("salary_band"), F.col("salary_band__dup1"), F.lit("Unknown"))
    )
else:
    best = best.withColumn(
        "salary_band_final",
        F.coalesce(F.col("salary_band"), F.lit("Unknown"))
    )


#  GENERATE EXPLANATIONS (Explainable AI output)

In [16]:
# Build 3-6 bullet reasons from evidence (Spark-only string building)
best = best.withColumn(
    "reason_1",
    F.concat(F.lit("Career Fit ranked this role #"), F.col("career_rank").cast("string"), F.lit(" (strong match)."))
)

best = best.withColumn(
    "reason_2",
    F.when(F.col("demand_trend") != "Unknown",
           F.concat(F.lit("Market demand trend: "), F.col("demand_trend"), F.lit("."))
          ).otherwise(F.lit("Market demand data not available; used neutral score."))
)

best = best.withColumn(
    "reason_3",
    F.when(F.col("salary_band_final") != "Unknown",
           F.concat(F.lit("Expected salary indicator: "), F.col("salary_band_final"), F.lit(".")))
     .otherwise(F.lit("Salary data not available; used neutral score."))
)


best = best.withColumn(
    "reason_4",
    F.concat(
        F.lit("Writing profile supports this role (writing fit score="),
        F.round(F.col("writing_fit_score"), 1).cast("string"),
        F.lit(").")
    )
)

best = best.withColumn(
    "reason_5",
    F.concat(
        F.lit("Final integrated score="),
        F.round(F.col("final_score"), 1).cast("string"),
        F.lit(" (weighted evidence integration).")
    )
)

# Combine reasons into one explanation text
best = best.withColumn(
    "final_explanation",
    F.concat_ws(" | ", "reason_1", "reason_2", "reason_3", "reason_4", "reason_5")
)

# FINAL OUTPUT TABLE

In [17]:
final_out = best.select(
    "Student_ID",
    F.col("role").alias("final_role"),
    "confidence_label",
    F.round("final_score", 2).alias("final_score"),
    F.round("career_fit_score", 1).alias("career_fit_score"),
    F.round("demand_score", 1).alias("demand_score"),
    F.round("salary_score", 1).alias("salary_score"),
    F.round("writing_fit_score", 1).alias("writing_fit_score"),
    "final_explanation"
)

final_out.show(20, truncate=False)

+----------+-----------------------+----------------+-----------+----------------+------------+------------+-----------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Student_ID|final_role             |confidence_label|final_score|career_fit_score|demand_score|salary_score|writing_fit_score|final_explanation                                                                                                                                                                                                                                                                    |
+----------+-----------------------+----------------+-----------+----------------+------------+------------+-----------------+----------------------------------------------------------------

# SAVE

In [18]:
(final_out.coalesce(1)
 .write.mode("overwrite")
 .option("header", True)
 .csv(OUT_FOLDER))

print(" AI Reasoning Layer output saved to:", OUT_FOLDER)

 AI Reasoning Layer output saved to: /content/drive/MyDrive/FYP1/Dataset/ai_reasoning_output
